# 01. Roamio Legacy Baseline Audit & Flaw Analysis

### Objective
In this notebook, we audit the original machine learning formulation in Roamio (the  module), diagnose why it fails both theoretically and empirically, and establish the mathematical foundation for why recommendation must be treated as **candidate retrieval and ranking**, not multi-class classification.

## 1. The Original Formulation: Country + Category → Destination

In the original codebase, the recommendation problem was formulated as:
15919\hat{y} = 	ext{LogisticRegression}(	ext{OneHot}(	ext{Country}), 	ext{OneHot}(	ext{Category}))15919
where $ is the **destination name** itself.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Load raw destinations data
df = pd.read_excel("attached_assets/destinations.xlsx")
print(f"Total dataset rows: {len(df)}")
print(f"Unique destination names: {df['Destination'].nunique()}")
print(f"Average samples per class: {len(df) / df['Destination'].nunique():.2f}")

## 2. Empirical Verification of Failure
Let us train the exact model and evaluate its accuracy on a held-out test split.

In [ ]:
X = df[['Country', 'Category']]
y = df['Destination']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
encoder = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), ['Country', 'Category'])])
clf = Pipeline([('enc', encoder), ('lr', LogisticRegression(max_iter=1000))])
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print(f"Test Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")

## 3. Why Did This Happen? Theoretical Diagnosis

1. **Support Deficit**: Logistic Regression requires dozens of examples per class to estimate separating hyperplanes. Here,  pprox K$ (167 training points, 167 classes).
2. **Disjoint Class Overlap**: Destinations appearing in the test set may have zero training examples in the training set.
3. **Conceptual Error**: Recommending travel is not a single-label prediction task. A user looking for a "Beach in Asia" does not have exactly one correct answer; there exists an ordered set of relevant candidates with varying degrees of affinity, affordability, and seasonal suitability.